# Введение в нейронные сети. Многослойный перцептрон

**AI1 · Neural Networks** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

В предыдущем разделе вы познакомились с самой простой моделью нейронной сети — однослойным перцептроном, который представляет собой линейную модель для двухклассовой классификации.

В этом разделе мы расширим эту модель, чтобы создать более гибкую структуру, которая позволит:

* выполнять **многоклассовую классификацию** в дополнение к двухклассовой,
* решать задачи **регрессии** в дополнение к классификации,
* разделять классы, которые не являются линейно разделимыми.

Мы также разработаем собственный модульный фреймворк на Python, который позволит нам строить различные архитектуры нейронных сетей.



## Формализация задачи машинного обучения

Начнем с формализации задачи машинного обучения. Предположим, у нас есть обучающая выборка **X** с метками **Y**, и нам нужно построить модель *f*, которая будет делать наиболее точные предсказания. Качество предсказаний измеряется с помощью **функции потерь** ℒ. Часто используются следующие функции потерь:

* Для задачи регрессии, когда нужно предсказать число, можно использовать **абсолютную ошибку** ∑<sub>i</sub>|f(x<sup>(i)</sup>)-y<sup>(i)</sup>| или **квадратичную ошибку** ∑<sub>i</sub>(f(x<sup>(i)</sup>)-y<sup>(i)</sup>)<sup>2</sup>.
* Для классификации используется **0-1 потеря** (она считает долю ошибок модели, то есть равна 1 − **точность**) или **логистическая потеря**.

Для однослойного перцептрона функция *f* определялась как линейная функция *f(x)=wx+b* (где *w* — это матрица весов, *x* — вектор входных признаков, а *b* — вектор смещений). Для различных архитектур нейронных сетей эта функция может принимать более сложную форму.

> В случае классификации часто желательно получить вероятности соответствующих классов на выходе сети. Чтобы преобразовать произвольные числа в вероятности (например, нормализовать выходные данные), часто используется функция **softmax** σ, и функция *f* становится *f(x)=σ(wx+b)*.

В определении *f* выше, *w* и *b* называются **параметрами** θ=⟨*w,b*⟩. Имея набор данных ⟨**X**,**Y**⟩, мы можем вычислить общую ошибку на всем наборе данных как функцию параметров θ.

> ✅ **Цель обучения нейронной сети — минимизировать ошибку, изменяя параметры θ**



## Оптимизация методом градиентного спуска

Существует известный метод оптимизации функций, называемый **градиентным спуском**. Идея заключается в том, что мы можем вычислить производную (в многомерном случае — **градиент**) функции потерь по параметрам и изменять параметры так, чтобы ошибка уменьшалась. Это можно формализовать следующим образом:

* Инициализировать параметры случайными значениями w<sup>(0)</sup>, b<sup>(0)</sup>.
* Повторять следующие шаги много раз:
    - w<sup>(i+1)</sup> = w<sup>(i)</sup>-η∂ℒ/∂w
    - b<sup>(i+1)</sup> = b<sup>(i)</sup>-η∂ℒ/∂b

Во время обучения шаги оптимизации предполагается вычислять с учетом всего набора данных (помните, что функция потерь вычисляется как сумма по всем обучающим примерам). Однако на практике мы берем небольшие части набора данных, называемые **минибатчами**, и вычисляем градиенты на основе подмножества данных. Поскольку подмножество выбирается случайно каждый раз, такой метод называется **стохастическим градиентным спуском** (SGD).



## Многослойные перцептроны и обратное распространение ошибки

Однослойная сеть, как мы видели выше, способна классифицировать линейно разделимые классы. Чтобы построить более сложную модель, мы можем объединить несколько слоев сети. Математически это означает, что функция *f* будет иметь более сложную форму и вычисляться в несколько шагов:
* z<sub>1</sub>=w<sub>1</sub>x+b<sub>1</sub>
* z<sub>2</sub>=w<sub>2</sub>α(z<sub>1</sub>)+b<sub>2</sub>
* f = σ(z<sub>2</sub>)

Здесь α — это **нелинейная функция активации**, σ — функция softmax, а параметры θ=<*w<sub>1</sub>,b<sub>1</sub>,w<sub>2</sub>,b<sub>2</sub>*>.

Алгоритм градиентного спуска остается тем же, но вычисление градиентов становится сложнее. Используя правило дифференцирования сложной функции, мы можем вычислить производные следующим образом:

* ∂ℒ/∂w<sub>2</sub> = (∂ℒ/∂σ)(∂σ/∂z<sub>2</sub>)(∂z<sub>2</sub>/∂w<sub>2</sub>)
* ∂ℒ/∂w<sub>1</sub> = (∂ℒ/∂σ)(∂σ/∂z<sub>2</sub>)(∂z<sub>2</sub>/∂α)(∂α/∂z<sub>1</sub>)(∂z<sub>1</sub>/∂w<sub>1</sub>)

> ✅ Правило дифференцирования сложной функции используется для вычисления производных функции потерь по параметрам.

Обратите внимание, что левая часть всех этих выражений одинакова, и поэтому мы можем эффективно вычислять производные, начиная с функции потерь и двигаясь "назад" по вычислительному графу. Таким образом, метод обучения многослойного перцептрона называется **обратным распространением ошибки** (backpropagation), или просто 'backprop'.

![граф вычислений](ms-assets/ai/ComputeGraphGrad.4626252c0de03507.webp)

> TODO: ссылка на источник изображения

> ✅ Мы рассмотрим обратное распространение ошибки гораздо подробнее в нашем примере с ноутбуком.



## Заключение

В этом уроке мы создали собственную библиотеку для работы с нейронными сетями и использовали ее для простой задачи классификации в двумерном пространстве.



## Задание

В сопровождающем ноутбуке вы реализуете собственный фреймворк для построения и обучения многослойных перцептронов. Вы сможете подробно изучить, как работают современные нейронные сети.

Перейдите к ноутбуку [OwnFramework](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/OwnFramework.ipynb) и выполните задания.



## Обзор и самостоятельное изучение

Обратное распространение ошибки — это распространенный алгоритм в области ИИ и машинного обучения, который стоит изучить [подробнее](https://wikipedia.org/wiki/Backpropagation).



## Задание

В этой лабораторной работе вам предстоит использовать фреймворк, который вы создали в этом уроке, для решения задачи классификации рукописных цифр из набора данных MNIST.

* [Инструкции](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/lab/README.md)
* [Ноутбук](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/lab/MyFW_MNIST.ipynb)



---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework)
- [Учебник ML (ШАД): нейронные сети](https://contest.yandex.ru/tracks/ml/deep-learning-introduction/neural-networks)
- [Математика для анализа данных: матанализ и градиенты](https://contest.yandex.ru/tracks/math/mathematical-analysis/chapter-overview)
- [Математика для анализа данных: линейная алгебра](https://contest.yandex.ru/tracks/math/linear-algebra/chapter-overview)
- [Учебник ML (ШАД): оптимизация](https://contest.yandex.ru/tracks/ml/optimization-in-machine-learning/optimization-in-machine-learning)